<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Surface_Curvature_Test_Hessian_Matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Professional Visualization of Multivariate Curvature: The Hessian Matrix

## 1. Introduction
This project provides a high-fidelity 3D simulation of mathematical curvature using the Manim animation engine. The primary objective is to bridge the gap between abstract linear algebra and three-dimensional geometry by visualizing the **Hessian Matrix** and its role in the **Second Derivative Test**. By analyzing the local behavior of the function $f(x, y) = \sin(x)\cos(y)$, this animation demonstrates how second-order partial derivatives characterize critical points as local extrema or saddle points.

## 2. Theoretical Background

### 2.1 The Hessian Matrix
For a scalar-valued function $f: \mathbb{R}^2 \to \mathbb{R}$, the Hessian matrix $H(f)$ is the square matrix of second-order partial derivatives:

$$H = \begin{bmatrix} f_{xx} & f_{xy} \\ f_{yx} & f_{yy} \end{bmatrix}$$

If the function is twice continuously differentiable ($C^2$), the mixed partials are equal ($f_{xy} = f_{yx}$), making the Hessian a symmetric matrix. This symmetry ensures that its eigenvalues are always real.

### 2.2 Local Curvature and Eigenvalues
The Hessian matrix describes the local quadratic approximation of a function near a point. The eigenvalues ($\lambda_1, \lambda_2$) of the Hessian represent the principal curvatures of the surface:
- **Positive Eigenvalues ($\lambda > 0$):** Indicate upward curvature (concave up).
- **Negative Eigenvalues ($\lambda < 0$):** Indicate downward curvature (concave down).

### 2.3 The Second Derivative Test
At a critical point (where the gradient $\nabla f = 0$), the nature of the point is determined by the definiteness of the Hessian:

1.  **Local Minimum:** If $H$ is positive definite ($\lambda_1, \lambda_2 > 0$), the surface curves upward in all directions.
2.  **Local Maximum:** If $H$ is negative definite ($\lambda_1, \lambda_2 < 0$), the surface curves downward in all directions.
3.  **Saddle Point:** If $H$ is indefinite ($\lambda_1 > 0$ and $\lambda_2 < 0$), the surface curves upward in one principal direction and downward in another.

## 3. Implementation Details
The simulation utilizes `Manim` to render a 3D surface and dynamically updates mathematical annotations. The camera is panned to maintain visibility of the mathematical expressions (Hessian components) while highlighting specific geometric features with curvature rings and arcs.

In [ ]:
!sudo apt update && sudo apt install -y libcairo2-dev libpango1.0-dev ffmpeg freeglut3-dev pkg-config libpng-dev libjpeg-dev libffi-dev texlive-latex-base texlive-fonts-recommended texlive-fonts-extra texlive-latex-extra dvisvgm
!pip install --force-reinstall "numpy<2.0.0" "manim==0.18.0" "scipy<1.13.0"

In [19]:
import os
import numpy as np
from manim import *
from google.colab import files
from IPython.display import HTML
from base64 import b64encode

# Configure global settings for high quality rendering
config.pixel_height = 720
config.pixel_width = 1280
config.frame_rate = 30
config.verbosity = "WARNING"

class HessianSimulation(ThreeDScene):
    """
    A professional 3D mathematical animation demonstrating the Second Derivative Test
    for functions of two variables, f(x, y).

    The Hessian Matrix (H) is the square matrix of second-order partial derivatives.
    This simulation visualizes how the eigenvalues (λ₁, λ₂) of the Hessian define
    local surface curvature (concavity) at critical points.

    Visual Definitions:
    - Local Maximum: Both λ₁, λ₂ < 0 (Surface curves down in all directions).
    - Saddle Point: λ₁ > 0 and λ₂ < 0 (Surface curves up in one axis, down in the other).
    """
    def construct(self):
        # Branding and Credits
        watermark = Text("@craftsandengineering", font_size=16, color=GRAY, fill_opacity=0.4).to_corner(DR, buff=0.2)
        author_text = Text("Mugambi Ndwiga", font_size=16, color=GRAY, fill_opacity=0.4).to_corner(DL, buff=0.2)
        self.add_fixed_in_frame_mobjects(watermark, author_text)

        def get_math_label(tex, description):
            """Helper to create a standardized annotation at the bottom of the screen."""
            group = VGroup(
                MathTex(tex, font_size=30, color=YELLOW),
                Text(description, font_size=22, color=WHITE)
            ).arrange(DOWN, buff=0.15).to_edge(DOWN, buff=0.6)
            return group

        # Descriptive Title
        title = Title("The Hessian Matrix \\& Surface Curvature", color=BLUE, font_size=36)
        self.add_fixed_in_frame_mobjects(title)

        # Surface Definition: f(x, y) = sin(x) * cos(y)
        axes = ThreeDAxes(axis_config={"stroke_width": 1})
        surface = Surface(
            lambda u, v: np.array([u, v, np.sin(u) * np.cos(v)]),
            u_range=[-3, 3], v_range=[-3, 3],
            resolution=(30, 30), fill_opacity=0.6, checkerboard_colors=[BLUE_D, BLUE_E]
        )

        # 1. Introduction: Display the Hessian Matrix structure
        hessian_formula = MathTex(
            r"H = \begin{bmatrix} f_{xx} & f_{xy} \\ f_{yx} & f_{yy} \end{bmatrix}",
            font_size=28
        ).to_edge(LEFT, buff=0.5).shift(UP * 1.2)
        explanation = Text("Hessian: A map of surface 'bending'", font_size=16, slant=ITALIC).next_to(hessian_formula, DOWN, buff=0.2)
        self.add_fixed_in_frame_mobjects(hessian_formula, explanation)

        # Set camera
        self.set_camera_orientation(phi=55 * DEGREES, theta=-35 * DEGREES)
        self.play(Create(axes), Create(surface), Write(hessian_formula), FadeIn(explanation), run_time=3)
        self.wait(1)

        # 2. Local Maximum Analysis: Visualizing Negative Definiteness
        max_label = get_math_label(
            r"f_{xx}, f_{yy} < 0 \implies \lambda_1, \lambda_2 < 0",
            "Local Peak: Negative second derivatives mean the surface bends DOWN."
        )
        self.add_fixed_in_frame_mobjects(max_label)
        peak_dot = Dot3D(point=[0, 0, 1], color=RED, radius=0.12)

        # Curvature rings to emphasize the peak shape
        c1 = Circle(radius=0.7, color=RED).rotate(90*DEGREES, axis=RIGHT).move_to([0,0,1])
        c2 = Circle(radius=0.7, color=RED).rotate(90*DEGREES, axis=UP).move_to([0,0,1])

        self.play(Write(max_label), Create(peak_dot), run_time=1.5, rate_func=smooth)
        self.play(Create(c1), Create(c2), run_time=2, rate_func=smooth)
        self.wait(3)
        self.play(FadeOut(c1), FadeOut(c2), FadeOut(max_label), run_time=1.5)

        # 3. Saddle Point Analysis: Visualizing Indefinite Matrices
        saddle_label = get_math_label(
            r"\text{Mixed signs in } H \implies \lambda_1 > 0, \lambda_2 < 0",
            "Saddle: One diagonal bends UP (positive), the other DOWN (negative)."
        )
        self.add_fixed_in_frame_mobjects(saddle_label)
        saddle_dot = Dot3D(point=[PI/2, PI/2, 0], color=YELLOW, radius=0.12)

        # Arcs showing contrasting directions of curvature
        curve_up = Arc(radius=0.9, start_angle=-PI/4, angle=PI/2, color=YELLOW).rotate(90*DEGREES, axis=RIGHT).move_to([PI/2, PI/2, 0.2])
        curve_down = Arc(radius=0.9, start_angle=PI-PI/4, angle=PI/2, color=YELLOW).rotate(90*DEGREES, axis=UP).move_to([PI/2, PI/2, -0.2])

        self.move_camera(theta=-20*DEGREES, run_time=2) # Adjusting pan again for better visibility of elements
        self.play(Write(saddle_label), ReplacementTransform(peak_dot, saddle_dot), run_time=2, rate_func=smooth)
        self.play(Create(curve_up), Create(curve_down), run_time=2, rate_func=smooth)
        self.wait(5)

# Instantiate and Render
scene = HessianSimulation()
scene.render()
video_path = "media/videos/720p30/HessianSimulation.mp4"

# Conversion for Colab Display
mp4 = open(video_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML(f"<video width=800 controls><source src='{data_url}' type='video/mp4'></video>"))
files.download(video_path)
print("Professional Render Complete.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Professional Render Complete.
